# Module 03 — Lesson 09
# Dot Product and Its Applications

---

> You've already been using the dot product for two lessons without naming it. Lesson 07's `magnitude()` secretly computed one. Every single entry of every matrix multiplication in Lesson 08 WAS one — a row dotted with a column. This lesson pulls that operation out into the open and shows you what it really means: a way to measure how much two vectors "agree."

The dot product turns out to answer a surprisingly wide range of real questions: How similar are two documents? Are two directions perpendicular? What's a model's prediction, given some weights and a data point? All of these reduce to the same simple computation — which is exactly why the dot product is often called **the engine inside linear regression and neural networks**.

---
## 🎯 Learning Objectives

By the end of this lesson, you will be able to:

- Compute the dot product of two vectors from scratch, algebraically
- Explain the dot product's geometric meaning: its relationship to the angle between two vectors
- Detect orthogonal (perpendicular) vectors using the dot product
- Compute cosine similarity and explain why it's preferred over the raw dot product for comparing direction
- Apply cosine similarity to a real similarity problem (comparing text documents)
- Explain how the dot product powers a linear model's prediction (weights · features)

---
## 1. What Is the Dot Product?

The **dot product** of two same-length vectors is a single number (a scalar): multiply each pair of corresponding components, then sum the results.

$$a \cdot b = \sum_{i=1}^{n} a_i b_i = a_1 b_1 + a_2 b_2 + \cdots + a_n b_n$$

Notice this is exactly what happened inside every entry of a matrix multiplication in Lesson 08 — and exactly what `magnitude()` computed in Lesson 07 (a vector dotted with itself).

In [ ]:
def dot_product(a, b):
    '''Compute the dot product of two same-length vectors.'''
    if len(a) != len(b):
        raise ValueError(f"Cannot dot vectors of different dimensions: {len(a)} vs {len(b)}")
    return sum(ai * bi for ai, bi in zip(a, b))


a = [1, 2, 3]
b = [4, 5, 6]

result = dot_product(a, b)
print(f"a = {a}")
print(f"b = {b}")
print(f"a . b = 1*4 + 2*5 + 3*6 = {result}")
print()
print("The result is a single SCALAR, not a vector — a dot product always")
print("collapses two vectors down to one number.")

---
## 2. Revisiting Magnitude: v · v = ||v||²

Lesson 07 defined magnitude as $\|v\| = \sqrt{v_1^2 + \cdots + v_n^2}$. Dotting a vector with **itself** gives exactly the sum of squares inside that square root:

$$v \cdot v = \|v\|^2 \quad \Longrightarrow \quad \|v\| = \sqrt{v \cdot v}$$

In [ ]:
import math

def magnitude(v):
    '''Euclidean length of a vector, now built directly from the dot product.'''
    return math.sqrt(dot_product(v, v))


v = [3, 4]
print(f"v . v          = {dot_product(v, v)}")
print(f"sqrt(v . v)    = {magnitude(v)}")
print(f"Lesson 07's answer for ||[3,4]|| was also 5.0 -- same formula, different name.")

---
## 3. The Geometric Meaning — Angle Between Vectors

The dot product has a second, equivalent definition involving the **angle** θ between two vectors:

$$a \cdot b = \|a\| \, \|b\| \, \cos(\theta)$$

Rearranging gives us a way to compute the angle between any two vectors, in any number of dimensions:

$$\theta = \arccos\left(\frac{a \cdot b}{\|a\| \, \|b\|}\right)$$

In [ ]:
def angle_between(a, b):
    '''Angle (in degrees) between two vectors, using the dot product formula.'''
    cos_theta = dot_product(a, b) / (magnitude(a) * magnitude(b))
    cos_theta = max(-1.0, min(1.0, cos_theta))   # clamp for floating-point safety
    return math.degrees(math.acos(cos_theta))


# Three vectors pointing in different directions
east      = [1, 0]
northeast = [1, 1]
north     = [0, 1]

print(f"Angle between east and northeast : {angle_between(east, northeast):.1f} degrees")
print(f"Angle between east and north      : {angle_between(east, north):.1f} degrees")
print(f"Angle between east and east       : {angle_between(east, east):.1f} degrees")
print()
print("The dot product secretly encodes direction information -- two vectors")
print("pointing the same way have angle 0, perpendicular vectors have angle 90.")

---
## 4. Orthogonal (Perpendicular) Vectors

When $\theta = 90°$, $\cos(\theta) = 0$, which means $a \cdot b = 0$. Two vectors with a **dot product of zero** are called **orthogonal**. This is a fast way to check perpendicularity without computing any angle at all.

In [ ]:
def is_orthogonal(a, b, tol=1e-9):
    '''True if a and b are perpendicular (dot product is ~0).'''
    return abs(dot_product(a, b)) < tol


pairs = [
    ([1, 0], [0, 1]),     # classic perpendicular axes
    ([2, 3], [3, -2]),    # perpendicular, different scale
    ([1, 1], [1, 0]),     # NOT perpendicular
]

for v1, v2 in pairs:
    orth = is_orthogonal(v1, v2)
    print(f"{v1} . {v2} = {dot_product(v1, v2):>3}   orthogonal? {orth}")

---
## 5. Cosine Similarity — Comparing Direction, Not Magnitude

The raw dot product is affected by BOTH the angle between vectors AND their magnitudes — a problem when you want to compare *direction/shape* only (e.g., "do these two customers like similar movies?" regardless of how many movies each rated). **Cosine similarity** divides out the magnitudes, leaving only the angle information:

$$\text{cosine\_similarity}(a, b) = \frac{a \cdot b}{\|a\| \, \|b\|} = \cos(\theta)$$

It always falls in the range **[-1, 1]**: 1 means identical direction, 0 means orthogonal, -1 means exactly opposite directions.

In [ ]:
def cosine_similarity(a, b):
    '''Cosine of the angle between a and b -- direction similarity, ignoring scale.'''
    mag_a, mag_b = magnitude(a), magnitude(b)
    if mag_a == 0 or mag_b == 0:
        raise ValueError("Cosine similarity is undefined when a vector has zero magnitude")
    return dot_product(a, b) / (mag_a * mag_b)


# Two users' movie ratings, on a scale of 1-5, for the same 4 movies
alice = [5, 4, 1, 5]    # rates enthusiastically
bob   = [1, 1, 0, 1]    # rates the SAME pattern, but much more conservatively

raw_dot = dot_product(alice, bob)
cos_sim = cosine_similarity(alice, bob)

print(f"Alice's ratings : {alice}")
print(f"Bob's ratings   : {bob}")
print()
print(f"Raw dot product   : {raw_dot}    <- doesn't reveal much on its own")
print(f"Cosine similarity : {cos_sim:.4f}  <- very close to 1: nearly IDENTICAL taste")
print()
print("Alice and Bob rate movies in the exact same RELATIVE pattern (loved movie 1,")
print("hated movie 3) even though their scales differ wildly. Cosine similarity")
print("sees past the scale difference; the raw dot product does not.")

---
## 6. Applied: Comparing Text Documents

A classic use of cosine similarity: turn text into vectors (word counts) and measure how similar two pieces of text are. This is the same core idea behind search engines ranking results and spam filters comparing emails.

In [ ]:
def word_count_vector(text, vocabulary):
    '''Turn text into a vector of word counts, one count per vocabulary word.'''
    words = text.lower().split()
    return [words.count(w) for w in vocabulary]


doc1 = "the cat sat on the mat"
doc2 = "the dog sat on the rug"
doc3 = "quantum physics explains subatomic particle behavior"

vocabulary = sorted(set(doc1.split()) | set(doc2.split()) | set(doc3.split()))

v1 = word_count_vector(doc1, vocabulary)
v2 = word_count_vector(doc2, vocabulary)
v3 = word_count_vector(doc3, vocabulary)

print(f"Vocabulary: {vocabulary}")
print()
print(f"doc1 vector: {v1}")
print(f"doc2 vector: {v2}")
print(f"doc3 vector: {v3}")
print()
print(f"similarity(doc1, doc2) = {cosine_similarity(v1, v2):.4f}   <- both about pets on a surface")
print(f"similarity(doc1, doc3) = {cosine_similarity(v1, v3):.4f}   <- completely unrelated topics")

---
## 7. The Dot Product as a Weighted Sum — The Engine of Linear Models

Lesson 08's "weighted final grade" example (`students_scores @ weights`) was really many dot products at once — one per student. This exact pattern, `weights · features + bias`, is how a **linear regression model** or a single **neuron/perceptron** computes its prediction.

In [ ]:
def predict(weights, bias, features):
    '''A linear model's prediction: dot(weights, features) + bias.'''
    return dot_product(weights, features) + bias


# A (toy, made-up) house price model:
# features = [size_in_100sqft, num_bedrooms, age_in_years]
weights = [45.0, 12.0, -1.5]   # learned "importance" of each feature
bias = 20.0                     # a base price offset

house_a = [18, 3, 10]   # 1800 sqft, 3 bed, 10 years old
house_b = [12, 2, 25]   # 1200 sqft, 2 bed, 25 years old

price_a = predict(weights, bias, house_a)
price_b = predict(weights, bias, house_b)

print(f"Predicted price for house A: {price_a:.1f} (lakhs)")
print(f"Predicted price for house B: {price_b:.1f} (lakhs)")
print()
print("This IS linear regression's prediction formula. Module 11 will show you")
print("how the WEIGHTS themselves get learned from data -- but the prediction")
print("step, once you have weights, is always just this one dot product.")

---
## 8. Verification: Consistency Across Lessons

In [ ]:
import math

# Verify dot_product-based magnitude matches Lesson 07's math.hypot-based version
v = [3, 4, 12]
our_magnitude = math.sqrt(dot_product(v, v))
stdlib_magnitude = math.hypot(*v)
print(f"magnitude via dot product : {our_magnitude:.6f}")
print(f"magnitude via math.hypot  : {stdlib_magnitude:.6f}")
print(f"Match: {abs(our_magnitude - stdlib_magnitude) < 1e-9}")
print()

# Verify angle_between + cosine_similarity are consistent with each other
a, b = [3, 1], [1, 2]
theta = angle_between(a, b)
cos_sim = cosine_similarity(a, b)
print(f"angle_between(a, b)             = {theta:.4f} degrees")
print(f"cos(angle_between(a, b))        = {math.cos(math.radians(theta)):.4f}")
print(f"cosine_similarity(a, b)         = {cos_sim:.4f}")
print(f"Match: {abs(math.cos(math.radians(theta)) - cos_sim) < 1e-9}")

---
## ⚠️ Common Mistakes

In [ ]:
# -- Mistake 1: Confusing dot product (a scalar) with elementwise product (a vector)

a, b = [1, 2, 3], [4, 5, 6]
elementwise = [ai * bi for ai, bi in zip(a, b)]   # a VECTOR
dot = dot_product(a, b)                            # a SCALAR

print("Mistake 1 — Dot product returns a SCALAR, not a vector:")
print(f"  elementwise products : {elementwise}   <- still a vector, 3 numbers")
print(f"  dot_product(a, b)    : {dot}            <- ONE number (their sum)")
print()
print("  The dot product IS 'multiply elementwise, then sum.' Stopping before")
print("  the sum gives you something else entirely.")

In [ ]:
# -- Mistake 2: Forgetting vectors must be the same dimension -----------------

a = [1, 2, 3]
b = [1, 2]

print("Mistake 2 — Mismatched dimensions:")
try:
    dot_product(a, b)
except ValueError as e:
    print(f"  Caught error: {e}")
print()
print("  Just like vector addition, the dot product requires both vectors to")
print("  have the exact same number of components.")

In [ ]:
# -- Mistake 3: Trusting the RAW dot product as a similarity score ------------

short_review = [1, 1, 0]     # a brief but very positive review, few words
long_review  = [5, 5, 0]     # a longer positive review, same RATIO of praise

raw_dot = dot_product(short_review, long_review)
cos_sim = cosine_similarity(short_review, long_review)

print("Mistake 3 — Raw dot product is skewed by magnitude, not just direction:")
print(f"  raw dot product     = {raw_dot}")
print(f"  cosine similarity   = {cos_sim:.4f}")
print()
print("  A bigger raw dot product does NOT necessarily mean 'more similar' --")
print("  it can just mean 'bigger vectors.' When comparing DIRECTION/PATTERN")
print("  (not scale), always use cosine similarity, not the raw dot product.")

In [ ]:
# -- Mistake 4: Assuming cosine similarity is always between 0 and 1 ----------

opposite_a = [1, 2, 3]
opposite_b = [-1, -2, -3]   # exactly the opposite direction

cos_sim = cosine_similarity(opposite_a, opposite_b)

print("Mistake 4 — Cosine similarity ranges from -1 to 1, NOT 0 to 1:")
print(f"  cosine_similarity({opposite_a}, {opposite_b}) = {cos_sim:.4f}")
print()
print("  A value of -1 means the vectors point in EXACTLY opposite directions --")
print("  this is a valid, meaningful result, not an error. Don't assume cosine")
print("  similarity behaves like a probability (which IS bounded to [0, 1]).")

---
## ✏️ Practice Exercises

### Exercise 1 — Starter: Dot Product By Hand and By Code

Compute the dot product of `a = [2, -1, 4]` and `b = [3, 5, -2]` by hand on paper first, then verify with `dot_product()`.

In [ ]:
a = [2, -1, 4]
b = [3, 5, -2]
# Your code here

### Exercise 2 — Verify v · v = ||v||²

For each vector in the list below, verify that `dot_product(v, v)` equals `magnitude(v) ** 2` (within floating-point tolerance).

```python
test_vectors = [[3, 4], [1, 1, 1], [5, -12], [0, 0, 0, 1]]
```

In [ ]:
test_vectors = [[3, 4], [1, 1, 1], [5, -12], [0, 0, 0, 1]]
# Your code here

### Exercise 3 — Find All Orthogonal Pairs

Given the list of vectors below, use `is_orthogonal()` to check every possible pair and print which pairs are perpendicular.

```python
vectors = {
    "v1": [1, 0, 0],
    "v2": [0, 1, 0],
    "v3": [1, 1, 0],
    "v4": [0, 0, 5],
    "v5": [2, -2, 0],
}
```

In [ ]:
vectors = {
    "v1": [1, 0, 0],
    "v2": [0, 1, 0],
    "v3": [1, 1, 0],
    "v4": [0, 0, 5],
    "v5": [2, -2, 0],
}
# Your code here

### Exercise 4 — Most Similar Product Profile

Each product below is a vector of feature ratings `[eco_friendliness, price_value, durability, style]` on a 1-10 scale. Using `cosine_similarity()`, find which OTHER product is most similar in "profile shape" to `target_product` (they don't need similar raw scores, just a similar pattern).

```python
target_product = [8, 3, 9, 6]
catalog = {
    "Product X": [4, 8, 5, 3],
    "Product Y": [7, 2, 8, 5],
    "Product Z": [2, 9, 3, 8],
}
```

In [ ]:
target_product = [8, 3, 9, 6]
catalog = {
    "Product X": [4, 8, 5, 3],
    "Product Y": [7, 2, 8, 5],
    "Product Z": [2, 9, 3, 8],
}
# Your code here

### Exercise 5 — (Challenge) A Tiny Perceptron

Write `perceptron_predict(weights, bias, features)` using `dot_product()`, that returns `1` if `dot(weights, features) + bias > 0`, else `0` (a simplified binary classifier, the building block of a neural network — you'll build on this in Module 12). Test it on a toy "pass/fail" dataset:

```python
# features = [hours_studied, attendance_pct] (simplified/scaled)
weights, bias = [0.5, 0.3], -30
students = [[10, 90], [2, 40], [6, 70], [1, 30]]
```

In [ ]:
weights, bias = [0.5, 0.3], -30
students = [[10, 90], [2, 40], [6, 70], [1, 30]]

def perceptron_predict(weights, bias, features):
    # Your code here
    pass

for s in students:
    print(s, perceptron_predict(weights, bias, s))

---
## 📌 Key Takeaways

- **The dot product — "multiply corresponding components, then sum" — is the single computation underneath magnitude, distance, and matrix multiplication, and now you've named and mastered it directly.** It collapses two vectors of any dimension down to one scalar.

- **Cosine similarity (the dot product divided by both magnitudes) measures direction/pattern while ignoring scale, and it's the standard tool for comparing text documents, user preferences, and recommendation profiles.** Remember it ranges from -1 to 1, not 0 to 1 — negative values are meaningful, not errors.

- **`weights · features + bias` is the prediction formula behind linear regression and a single neural network neuron.** Everything from Lesson 07's vectors through Lesson 08's matrices converges here: a model's prediction is just a dot product, learned weights are just a vector, and a whole dataset's predictions are just a matrix multiplication.

---
## What's Next?

**Module 03 Project — Marks Analysis: Statistical Report Generator**
You've now built the complete mathematical toolkit for this course: descriptive statistics (Lessons 01–03), probability and distributions (Lessons 04–06), and linear algebra (Lessons 07–09). The module project brings it all together — you'll load a real marks dataset and produce a full statistical report: per-subject descriptive stats, department comparisons, outlier detection, grade distributions, and a correlation between study hours and scores.